# Adjoint method — toy verification

Self-contained sanity check of the discrete-time adjoint formulae from `main.tex` against PyTorch autograd, on a tiny flow-matching velocity field $v_\theta(a, t)$.

Two cases:

1. **Terminal-loss adjoint.** Compute $\nabla_\theta \mathcal{L}(a_N)$ where $a_N$ is the endpoint of an Euler rollout of $\dot a = v_\theta(a, t)$ on $[0, 1]$. The discrete adjoint runs a backward recurrence over the saved trajectory.
2. **Augmented adjoint for log-density.** Compute $\nabla_\theta \ell_N$ where $\ell_N = \log \pi_\theta(a_N) = \log p_0(a_0) - \int_0^1 \nabla_a\!\cdot v_\theta\,dt$, using the augmented state $(a, \ell)$. The adjoint pair $(\mu, \nu)$ collapses to $\nu \equiv 1$ and a forced backward ODE for $\mu$.

In both cases we verify the adjoint matches autograd to machine precision in fp64. Action dim is `2`, so divergence is computed *exactly* with two backward passes (no Hutchinson). The same recipe extends directly to `unified_flowmatching_sampler` in policy mode — there the integrated state is the joint future $(a, o)$ window and the conditioning (context obs + actions + sigma indices) is exogenous and held fixed.

In [2]:
import math
import torch
import torch.nn as nn

torch.manual_seed(0)
DTYPE = torch.float64                  # fp64 → grad mismatches at ~1e-16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Toy velocity field

Tiny MLP $v_\theta : \mathbb{R}^d \times [0,1] \to \mathbb{R}^d$. We use `DIM=2` so the divergence $\nabla_a\!\cdot v_\theta$ is just the sum of two scalar partials — computable exactly with two backward passes, no estimator needed.

In [3]:
class TinyVelocity(nn.Module):
    def __init__(self, dim=2, hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim + 1, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, dim),
        )

    def forward(self, a, t):
        # a: (B, d), t: scalar tensor or float
        t_vec = torch.full((a.shape[0], 1), float(t), device=a.device, dtype=a.dtype)
        return self.net(torch.cat([a, t_vec], dim=-1))


DIM = 2
N_STEPS = 32
B = 4    # batch of independent trajectories

vmodel = TinyVelocity(dim=DIM).to(device=device, dtype=DTYPE)
a0 = torch.randn(B, DIM, device=device, dtype=DTYPE)
ts = torch.linspace(0.0, 1.0, N_STEPS + 1, device=device, dtype=DTYPE)

n_params = sum(p.numel() for p in vmodel.parameters())
print(f"Velocity field: {n_params} parameters, action dim={DIM}, {N_STEPS} Euler steps, batch={B}")

Velocity field: 370 parameters, action dim=2, 32 Euler steps, batch=4


## Section 1 — Terminal-loss adjoint

Discrete map (forward Euler):
$$F_k(a, \theta) = a + \Delta t\, v_\theta(a, t_k), \qquad k=0,\dots,N-1.$$

Terminal loss: $\mathcal{L}(a_N) = \tfrac{1}{2}\,\overline{\lVert a_N \rVert^2}$ (mean over batch and dimension — a vanilla "shrink the endpoint" objective).

Discrete adjoint:
$$\lambda_N = \frac{\partial \mathcal{L}}{\partial a_N}, \qquad \lambda_k = \lambda_{k+1} + \Delta t\, J_v^T \lambda_{k+1}, \qquad \nabla_\theta \mathcal{L} = \sum_{k=0}^{N-1} \Delta t\, \lambda_{k+1}^T \frac{\partial v}{\partial \theta}\bigg|_{(a_k, t_k)}.$$

Each backward step is **one VJP through $v_\theta$ wrt both $a$ and $\theta$ in a single call** — no activation cache across the $N$ Euler steps.

In [4]:
@torch.no_grad()
def forward_euler(v_theta, a0, ts):
    """Forward Euler rollout, returning the saved trajectory (no autograd graph)."""
    a = a0.clone()
    traj = [a.clone()]
    for k in range(len(ts) - 1):
        dt = ts[k + 1] - ts[k]
        v = v_theta(a, ts[k])
        a = a + dt * v
        traj.append(a.clone())
    return traj


def terminal_loss(a):
    return 0.5 * (a ** 2).mean()


traj = forward_euler(vmodel, a0, ts)
L_val = terminal_loss(traj[-1]).item()
print(f"a_0 mean = {traj[0].mean().item():+.4f}")
print(f"a_N mean = {traj[-1].mean().item():+.4f}")
print(f"L(a_N)   = {L_val:.6f}")

a_0 mean = -0.0615
a_N mean = -0.1661
L(a_N)   = 0.388302


In [5]:
def adjoint_terminal_grad(v_theta, traj, ts, loss_fn):
    """Discrete adjoint backward — returns dL/dtheta as a list aligned with v_theta.parameters()."""
    a_N = traj[-1].detach().clone().requires_grad_(True)
    L = loss_fn(a_N)
    (lam,) = torch.autograd.grad(L, a_N)        # lambda_N = dL/da_N

    params = list(v_theta.parameters())
    grads = [torch.zeros_like(p) for p in params]

    for k in reversed(range(len(ts) - 1)):
        dt = (ts[k + 1] - ts[k]).item()
        a_k = traj[k].detach().clone().requires_grad_(True)
        v = v_theta(a_k, ts[k])
        # One VJP yields (J_v^T lam) for the state and (∂v/∂θ)^T lam for the parameters.
        out = torch.autograd.grad(v, [a_k] + params, grad_outputs=dt * lam)
        grad_a, grads_p = out[0], out[1:]
        lam = lam + grad_a                       # lambda_k = (I + dt J_v^T) lambda_{k+1}
        for g, gp in zip(grads, grads_p):
            g.add_(gp)
    return grads


grads_adj = adjoint_terminal_grad(vmodel, traj, ts, terminal_loss)
print("Adjoint backward complete.")

Adjoint backward complete.


In [6]:
# Ground truth: backprop through the entire Euler rollout.
def autograd_terminal_grad(v_theta, a0, ts, loss_fn):
    a = a0.clone()
    for k in range(len(ts) - 1):
        dt = ts[k + 1] - ts[k]
        v = v_theta(a, ts[k])
        a = a + dt * v
    L = loss_fn(a)
    return list(torch.autograd.grad(L, list(v_theta.parameters())))


grads_naive = autograd_terminal_grad(vmodel, a0, ts, terminal_loss)

print(f"{'param':25s}  {'max|adj - naive|':>18s}  {'rel err':>12s}")
print("-" * 60)
for (name, _), ga, gn in zip(vmodel.named_parameters(), grads_adj, grads_naive):
    diff = (ga - gn).abs().max().item()
    rel = diff / max(gn.abs().max().item(), 1e-12)
    print(f"{name:25s}  {diff:18.2e}  {rel:12.2e}")

param                        max|adj - naive|       rel err
------------------------------------------------------------
net.0.weight                         0.00e+00      0.00e+00
net.0.bias                           0.00e+00      0.00e+00
net.2.weight                         0.00e+00      0.00e+00
net.2.bias                           0.00e+00      0.00e+00
net.4.weight                         0.00e+00      0.00e+00
net.4.bias                           0.00e+00      0.00e+00


## Section 2 — Augmented adjoint for log-density

Augmented state $U_k = (a_k, \ell_k)$ with map
$$F_k(U, \theta) = \bigl(a + \Delta t\, v_\theta(a, t_k),\ \ell - \Delta t\, d_\theta(a, t_k)\bigr), \qquad d_\theta := \nabla_a\!\cdot v_\theta.$$

Initialise $\ell_0 = \log p_0(a_0)$ with $p_0 = \mathcal{N}(0, I)$, so $\ell_N = \log \pi_\theta(a_N)$ exactly (instantaneous change-of-variables).

Adjoint pair $\Lambda = (\mu, \nu)$ for the terminal objective $\mathcal{L} = \sum_b \ell_N^{(b)}$:
$$\mu_N = 0, \quad \nu_N = 1; \qquad \mu_k = \mu_{k+1} + \Delta t\,J_v^T \mu_{k+1} - \Delta t\,(\nabla_a d_\theta)^T \nu_{k+1}, \qquad \nu_k = \nu_{k+1}.$$

Hence $\nu \equiv 1$ along the trajectory, and the parameter gradient picks up two terms:
$$\nabla_\theta \mathcal{L} = \sum_{k=0}^{N-1} \Delta t\,\Bigl(\mu_{k+1}^T\, \tfrac{\partial v_\theta}{\partial \theta} - \tfrac{\partial d_\theta}{\partial \theta}\Bigr)\bigg|_{(a_k, t_k)}.$$

Same Section-1 recurrence, plus a divergence-forcing term in $\mu$ and a $-\partial d/\partial \theta$ contribution in the parameter sum. We compute $\nabla_a d$ and $\partial d/\partial \theta$ exactly via nested autograd (these are second-order quantities in $v_\theta$).

In [ ]:
def divergence(v_theta, a, t, create_graph=False):
    """Exact divergence d = sum_i ∂v_i/∂a_i, per batch element. Returns shape (B,)."""
    v = v_theta(a, t)
    d = torch.zeros(a.shape[0], device=a.device, dtype=a.dtype)
    n = a.shape[-1]
    for i in range(n):
        gi, = torch.autograd.grad(
            v[..., i].sum(), a,
            create_graph=create_graph,
            retain_graph=True,                # keep graph alive for next coord
        )
        d = d + gi[..., i]
    return d


def log_p0(a):
    """Standard normal log-density."""
    d = a.shape[-1]
    return -0.5 * (a ** 2).sum(dim=-1) - 0.5 * d * math.log(2 * math.pi)

In [ ]:
def forward_augmented(v_theta, a0, ts):
    """Integrate the augmented (a, ell) system. No autograd graph retained."""
    a = a0.clone()
    ell = log_p0(a)
    traj_a = [a.clone()]
    traj_ell = [ell.clone()]
    for k in range(len(ts) - 1):
        dt = (ts[k + 1] - ts[k]).item()
        with torch.enable_grad():
            a_in = a.detach().clone().requires_grad_(True)
            v_k = v_theta(a_in, ts[k])
            d_k = divergence(v_theta, a_in, ts[k], create_graph=False)
        a = a + dt * v_k.detach()
        ell = ell - dt * d_k.detach()
        traj_a.append(a.clone())
        traj_ell.append(ell.clone())
    return traj_a, traj_ell


traj_a, traj_ell = forward_augmented(vmodel, a0, ts)
print("log pi(a_N) per trajectory:", traj_ell[-1].cpu().numpy())

In [ ]:
def adjoint_logpi_grad(v_theta, traj_a, ts):
    """Augmented discrete adjoint — returns d(sum_b ell_N^b)/dtheta."""
    B, d = traj_a[0].shape
    dev, dt0 = traj_a[0].device, traj_a[0].dtype
    mu = torch.zeros(B, d, device=dev, dtype=dt0)
    nu = torch.ones(B, device=dev, dtype=dt0)        # collapses to 1, kept explicit for clarity

    params = list(v_theta.parameters())
    grads = [torch.zeros_like(p) for p in params]

    for k in reversed(range(len(ts) - 1)):
        dt = (ts[k + 1] - ts[k]).item()

        # ---- v-VJP at (a_k, theta), cotangent dt * mu_{k+1} ----
        a_in = traj_a[k].detach().clone().requires_grad_(True)
        v = v_theta(a_in, ts[k])
        out_v = torch.autograd.grad(v, [a_in] + params, grad_outputs=dt * mu)
        ga_v, gp_v = out_v[0], out_v[1:]

        # ---- divergence-VJP at (a_k, theta), cotangent dt * nu_{k+1} ----
        # second-order: d depends on (∂v/∂a), so we VJP-through-a-Jacobian.
        # Some params (e.g. final bias) don't enter d at all → allow_unused=True.
        a_in2 = traj_a[k].detach().clone().requires_grad_(True)
        d_val = divergence(v_theta, a_in2, ts[k], create_graph=True)
        out_d = torch.autograd.grad(
            d_val, [a_in2] + params, grad_outputs=dt * nu, allow_unused=True,
        )
        ga_d = out_d[0]
        gp_d = [g if g is not None else torch.zeros_like(p)
                for g, p in zip(out_d[1:], params)]

        # mu_k = (I + dt J_v^T) mu_{k+1} - dt (∇_a d)^T  nu_{k+1}
        mu = mu + ga_v - ga_d
        # nu unchanged

        # parameter-grad contribution: dt (μ^T ∂v/∂θ - ∂d/∂θ)
        for g, gv, gd in zip(grads, gp_v, gp_d):
            g.add_(gv - gd)
    return grads


grads_aug_adj = adjoint_logpi_grad(vmodel, traj_a, ts)
print("Augmented adjoint backward complete.")

In [ ]:
# Ground truth: backprop through the augmented forward (a₀ kept as a leaf so the
# inner divergence calls — which need autograd.grad(v, a, ...) — succeed at k=0).
def autograd_logpi_grad(v_theta, a0, ts):
    a = a0.detach().clone().requires_grad_(True)
    ell = log_p0(a)
    for k in range(len(ts) - 1):
        dt = (ts[k + 1] - ts[k]).item()
        v = v_theta(a, ts[k])
        d = divergence(v_theta, a, ts[k], create_graph=True)
        a = a + dt * v
        ell = ell - dt * d
    L = ell.sum()
    return list(torch.autograd.grad(L, list(v_theta.parameters())))


grads_aug_naive = autograd_logpi_grad(vmodel, a0, ts)

print(f"{'param':25s}  {'max|adj - naive|':>18s}  {'rel err':>12s}")
print("-" * 60)
for (name, _), ga, gn in zip(vmodel.named_parameters(), grads_aug_adj, grads_aug_naive):
    diff = (ga - gn).abs().max().item()
    rel = diff / max(gn.abs().max().item(), 1e-12)
    print(f"{name:25s}  {diff:18.2e}  {rel:12.2e}")

## Notes

**Why this matters for `unified_flowmatching_sampler`.**

In policy mode the per-step Euler update at [sampling.py:317-329](../dreamerv4uwm/sampling.py#L317-L329) is structurally identical to $F_k$ above, with two differences:

1. The state is the joint future window $U = (z_{\text{fut}},\ z_{\text{act, fut}})$. The denoiser produces both $\hat z, \hat z_{\text{act}}$; concatenate them and they play the role of $a$ here. The Jacobian $J_v$ is just the input-output Jacobian of `denoiser(...)` restricted to the future positions.
2. There's an extra $\frac{1}{1-\tau_k}$ scalar inside the Euler step (velocity reparameterisation). Pulls back through the adjoint cleanly — just multiplies the cotangent at step $k$.

The terminal-loss case (Section 1) maps directly onto things like *minimise mean reward predictor on the generated $(\hat o, \hat a)$* or *match a target action* — same $O(1)$-in-`num_diffusion_steps` memory win.

**Hutchinson for high-dim.** In Section 2 we computed exact divergence with one VJP per coordinate — fine for $d=2$, hopeless for the UWM joint state ($\sim 8\,\mathrm{k}$ dims). Replace
$$\nabla_a\!\cdot v_\theta \;\approx\; \xi^\top J_v\, \xi, \qquad \xi \sim \mathcal{N}(0, I)\ \text{or Rademacher}$$
which is one VJP per step at the cost of variance. Same trick used to train CNFs.

**Memory.** Naive autograd through the rollout caches $N$ copies of denoiser activations. The adjoint backward recomputes one forward + one VJP per step on the saved $\{U_k\}$ — $O(1)$ in $N$, at the cost of ~2× wall-clock vs the forward sampler. For the augmented (log-density) case the per-step backward is ~3× because of the second-order quantities; Hutchinson keeps that bounded.

**Boundary term.** $a_0 \sim \mathcal{N}(0, I)$ is sampled independently of $\theta$, so $\lambda_0^T\, da_0/d\theta = 0$ — no contribution from the initial-condition term, exactly as in the LaTeX example.